In [ ]:
import pandas as pd
from pathlib import Path
import os
import glob
import warnings
import numpy as np
from collections import Counter
from typing import List, Tuple

# Suppress warnings that might clutter the notebook output (consistent with Target)
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================================
# USER CONFIGURABLE PARAMETERS
# ==============================================================================

# --- File Paths and Patterns ---
# Root directory for finding input files.
SEARCH_ROOT_DIR = "."

# The pattern for the input XLSX files.
INPUT_FILE_PATTERN_GLOB = "deltaSASA_summary_repl_*.xlsx"
INPUT_SHEET_NAME = 'delta_sasa_raw'

# The Excel output file name pattern.
# FIX: Added the leading slash '/' to make the path absolute
OUTPUT_XLSX_PATTERN = "."

# --- Scoring Thresholds ---
MAGNITUDE_THRESHOLDS: List[Tuple[float, str, float]] = [
    (0.5, "major", 3), (0.3, "moderate", 2), (0.1, "minor", 1),
]

OCCUPANCY_THRESHOLDS: List[Tuple[float, str, float]] = [
    (0.7, "core", 3), (0.5, "persistent", 2), (0.3, "transient", 1),
]

RELIABILITY_THRESHOLDS: List[Tuple[float, str, float]] = [
    (3, "strict", 3), (2, "moderate", 2), (1, "permissive", 1),
]

# --- Normalization Constants ---
MAX_COMPONENT_SCORE = 0.33
MAX_MAGNITUDE_SCORE = 3
MAX_OCC_SCORE = 3
MAX_DEPTH_SCORE = 3
MAX_RELIABILITY_SCORE = 3
MIN_TOTAL_OCCUPANCY_FILTER = 0.3

# ==============================================================================
# HELPER FUNCTIONS (Flagging and scoring logic)
# ==============================================================================

def flag_magnitude(delta_sasa: float, thresholds: List[Tuple[float, str, float]]) -> tuple[str, float]:
    for threshold, label, score in thresholds:
        if delta_sasa >= threshold: return label, score
    return "none", 0.0

def calculate_total_occupancy(row: pd.Series) -> float:
    return row["occupancy_0.1_0.3"] + row["occupancy_0.3_0.5"] + row["occupancy_gt_0.5"]

def flag_total_occupancy(total_occ: float, thresholds: List[Tuple[float, str, float]]) -> tuple[str, float]:
    for threshold, label, score in thresholds:
        if total_occ >= threshold: return label, score
    return "none", 0.0

def flag_depth_contribution(row: pd.Series, total_occ: float) -> tuple[str, float]:
    if total_occ == 0: return "mixed", 1.5

    fractions = {
        "mild": row["occupancy_0.1_0.3"] / total_occ,
        "moderate": row["occupancy_0.3_0.5"] / total_occ,
        "strong": row["occupancy_gt_0.5"] / total_occ,
    }

    for key, frac in fractions.items():
        if frac >= 0.5: return key, {"mild": 1.0, "moderate": 2.0, "strong": 3.0}[key]

    return "mixed", 1.5

def flag_reliability(delta_sasa: float, std_dev: float, thresholds: List[Tuple[float, str, float]]) -> tuple[str, float]:
    if std_dev == 0 or np.isnan(std_dev) or std_dev is None: return "none", 0.0
    ratio = delta_sasa / std_dev
    for multiplier, label, score in thresholds:
        if ratio >= multiplier: return label, score
    return "none", 0.0

def process_dataframe(df: pd.DataFrame, delta_col: str, sd_col: str, 
                      mag_thresholds: List[Tuple[float, str, float]],
                      occ_thresholds: List[Tuple[float, str, float]],
                      rel_thresholds: List[Tuple[float, str, float]],
                      min_occ_filter: float) -> pd.DataFrame:
    """
    Processes and scores residues for a given ΔSASA metric (mean or median),
    taking all necessary constants as arguments.
    
    This function has been updated to include 'residue_pdb_resseq' in the
    base columns list for the final output order.
    """
    df = df.rename(columns={'resid': 'residue', 'sd': sd_col}, errors='ignore')
    df["total_occupancy"] = df.apply(calculate_total_occupancy, axis=1)

    df_filtered = df[
        (df[delta_col] >= 0.1)
        & (df[delta_col].abs() >= df[sd_col])
        & (df["total_occupancy"] >= min_occ_filter)
    ].copy()

    mag_flags, mag_scores, occ_flags, depth_flags = [], [], [], []
    persistence_scores, rel_flags, rel_scores, combined_flags, total_scores = [], [], [], [], []

    for _, row in df_filtered.iterrows():
        delta = row[delta_col]; sd = row[sd_col]; tot_occ = row["total_occupancy"]

        mag_flag, mag_raw = flag_magnitude(delta, mag_thresholds)
        mag_flags.append(mag_flag); mag_scores.append(mag_raw * MAX_COMPONENT_SCORE / MAX_MAGNITUDE_SCORE)

        occ_flag, occ_raw = flag_total_occupancy(tot_occ, occ_thresholds)
        occ_flags.append(occ_flag)

        depth_flag, depth_raw = flag_depth_contribution(row, tot_occ)
        depth_flags.append(depth_flag)

        persistence = (occ_raw + depth_raw) * MAX_COMPONENT_SCORE / (MAX_OCC_SCORE + MAX_DEPTH_SCORE)
        persistence_scores.append(persistence); combined_flags.append(f"{occ_flag}-{depth_flag}")

        rel_flag, rel_raw = flag_reliability(delta, sd, rel_thresholds)
        rel_flags.append(rel_flag); rel_scores.append(rel_raw * MAX_COMPONENT_SCORE / MAX_RELIABILITY_SCORE)

        total_scores.append(mag_scores[-1] + persistence_scores[-1] + rel_scores[-1])

    df_filtered[f"{delta_col}_magnitude_flag"] = mag_flags
    df_filtered[f"{delta_col}_magnitude_score"] = mag_scores
    df_filtered[f"{delta_col}_total_occ_flag"] = occ_flags
    df_filtered[f"{delta_col}_depth_flag"] = depth_flags
    df_filtered[f"{delta_col}_persistence_score"] = persistence_scores
    df_filtered[f"{delta_col}_combined_occ_flag"] = combined_flags
    df_filtered[f"{delta_col}_reliability_flag"] = rel_flags
    df_filtered[f"{delta_col}_reliability_score"] = rel_scores
    df_filtered["weighted_score_total"] = total_scores

    df_filtered = df_filtered.sort_values(
        ["chain", "weighted_score_total", delta_col], ascending=[True, False, False]
    )
    df_filtered["weighted_score_rank"] = df_filtered.groupby("chain").cumcount() + 1

    df_filtered.sort_values(
        by=["chain", "weighted_score_rank"], ascending=[True, True], inplace=True
    )

    drop_cols = ['mean_sasa_unbound_nm2', 'mean_sasa_bound_nm2', 'median_sasa_unbound_nm2', 'median_sasa_bound_nm2', "total_occupancy"]
    other_delta_col = "median_delta_sasa_nm2" if delta_col == "mean_delta_sasa_nm2" else "mean_delta_sasa_nm2"
    if other_delta_col in df_filtered.columns: drop_cols.append(other_delta_col)

    df_filtered.drop(columns=drop_cols, errors="ignore", inplace=True)

    # Base columns definition modified to insert the new column between "chain" and "resname"
    base_cols = [
        "chain",
        "residue_pdb_resseq", # NEW COLUMN ADDED
        "residue",
        "resname",
        delta_col,
        sd_col,
        "occupancy_0.1_0.3",
        "occupancy_0.3_0.5",
        "occupancy_gt_0.5"
    ]
    
    score_cols = ["weighted_score_total", "weighted_score_rank", f"{delta_col}_magnitude_score", f"{delta_col}_persistence_score", f"{delta_col}_reliability_score"]
    flag_cols = [f"{delta_col}_magnitude_flag", f"{delta_col}_total_occ_flag", f"{delta_col}_depth_flag", f"{delta_col}_combined_occ_flag", f"{delta_col}_reliability_flag"]

    return df_filtered[[c for c in base_cols + score_cols + flag_cols if c in df_filtered.columns]]


def run_sasa_scoring_analysis():
    """
    Main execution function. Traverses replica directories, loads the raw ΔSASA data,
    apply the scoring logic, and saves the flagged and ranked results.
    
    ADJUSTED: Finds all deltaSASA_summary_repl_*.xlsx files directly in SEARCH_ROOT_DIR.
    """
    search_path = Path(SEARCH_ROOT_DIR)
    
    # ADJUSTMENT 3: Find all matching XLSX files directly in the root directory
    input_files = sorted(
        search_path.glob(INPUT_FILE_PATTERN_GLOB)
    )

    if not input_files:
        print(f"Global ERROR ❌ - No files matching '{INPUT_FILE_PATTERN_GLOB}' found inside {SEARCH_ROOT_DIR}. Exiting.")
        return

    print(f"Global INFO - Found {len(input_files)} input files. Starting scoring analysis.")

    for input_xlsx_path in input_files:
        
        # Extract the replica number from the filename (e.g., 'deltaSASA_summary_repl_123.xlsx' -> '123')
        # Assuming the format is always *_repl_{repl}.xlsx
        try:
            filename_parts = input_xlsx_path.stem.split('_')
            repl_index = filename_parts.index('repl') + 1
            replica_num = filename_parts[repl_index]
        except ValueError:
            print(f"File Parsing WARNING ⚠️ - Could not determine replica number from filename: {input_xlsx_path.name}. Skipping.")
            continue


        output_xlsx_base = OUTPUT_XLSX_PATTERN.format(repl=replica_num)

        # Since the OUTPUT_XLSX_PATTERN is an absolute path, we'll use it directly
        output_xlsx_path = output_xlsx_base
        
        print(f"\n>>> Starting ΔSASA Scoring for Replica [{replica_num}] (Input: {input_xlsx_path.name})")

        # --- Step 1: Load Input Data ---
        try:
            df_raw = pd.read_excel(input_xlsx_path, sheet_name=INPUT_SHEET_NAME)
            # Ensure output directory exists before writing
            os.makedirs(Path(output_xlsx_path).parent, exist_ok=True) 
            print(f"Replica [{replica_num}] INFO - Loaded sheet '{INPUT_SHEET_NAME}' with {len(df_raw)} rows.")
        except FileNotFoundError:
            # This should ideally not happen since we globbed for existing files
            print(f"Replica [{replica_num}] WARNING ⚠️ - Skipping missing input file: {input_xlsx_path.name}")
            continue
        except Exception as e:
            print(f"Replica [{replica_num}] ERROR ❌ - Could not read input file: {e}. Skipping replica.")
            continue

        mean_delta_col = 'mean_delta_sasa_nm2'; median_delta_col = 'median_delta_sasa_nm2'; sd_col = 'sd_delta_sasa_nm2'

        # --- Step 2: Apply Scoring Logic to Mean and Median ΔSASA ---
        df_mean_scored = process_dataframe(df_raw.copy(), mean_delta_col, sd_col, 
                                           MAGNITUDE_THRESHOLDS, OCCUPANCY_THRESHOLDS, 
                                           RELIABILITY_THRESHOLDS, MIN_TOTAL_OCCUPANCY_FILTER)
        print(f"Replica [{replica_num}] INFO - Mean ΔSASA scored. Filtered rows: {len(df_mean_scored)}.")

        df_median_scored = process_dataframe(df_raw.copy(), median_delta_col, sd_col,
                                             MAGNITUDE_THRESHOLDS, OCCUPANCY_THRESHOLDS, 
                                             RELIABILITY_THRESHOLDS, MIN_TOTAL_OCCUPANCY_FILTER)
        print(f"Replica [{replica_num}] INFO - Median ΔSASA scored. Filtered rows: {len(df_median_scored)}.")


        # --- Step 3: Save results to Excel (Standardized to xlsxwriter) ---
        try:
            with pd.ExcelWriter(output_xlsx_path, engine='xlsxwriter') as writer:
                df_mean_scored.to_excel(writer, sheet_name="mean_delta_scored", index=False)
                df_median_scored.to_excel(writer, sheet_name="median_delta_scored", index=False)

            print(f"Replica [{replica_num}] SUCCESS ✅ - Results saved to {Path(output_xlsx_path).name} (mean_delta_scored + median_delta_scored sheets)")

        except Exception as e:
            csv_out_mean = output_xlsx_path.replace('.xlsx', '_mean_scored.csv')
            csv_out_median = output_xlsx_path.replace('.xlsx', '_median_scored.csv')
            df_mean_scored.to_csv(csv_out_mean, index=False)
            df_median_scored.to_csv(csv_out_median, index=False)

            print(f"Replica [{replica_num}] ERROR ❌ - Could not write XLSX file: {e}. Saved CSV fallback to {Path(csv_out_mean).name} and {Path(csv_out_median).name}")


# Execute the main function when the script runs
if __name__ == "__main__":
    run_sasa_scoring_analysis()